# 04 — End-to-End Testing: Fusion + Full Pipeline on Real Data

Loads the two branches trained on real data in `02_train_static_branch.ipynb` and
`03_train_dynamic_branch.ipynb`, and:

1. Re-confirms each branch's held-out test EER/AUC (the final "test" numbers).
2. Runs the full `sigverify.pipeline.inference.verify_signature` pipeline —
   preprocessing, both branches, cross-attention fusion, decision fusion,
   explainability — on real CEDAR (static) and MOBISIG (dynamic) samples.

**On fusion pairing.** CEDAR (offline scans) and MOBISIG (finger-drawn
touchscreen captures) are two independent public datasets with disjoint,
unrelated writer populations — neither pairs a static scan and a dynamic
capture of the *same physical signing event*. That pairing only exists in
datasets like DeepSignDB/e-BioSign, which require an access request this
session didn't chase down. So the fusion network below is trained on the
paired **synthetic** demo set (`sigverify.data.synthetic`), which shares the
same embedding geometry (unit-normalized vectors, similar genuine/forged
separation) — a defensible bridge for exercising the fusion *mechanism*
end-to-end, while the two branches' own accuracy numbers come entirely from
real data.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import torch
from torch.utils.data import DataLoader

from sigverify.data.datasets import (
    StaticSignatureTripletDataset, DynamicStrokeTripletDataset, split_writers, load_manifest,
)
from sigverify.models.static_branch import SiameseCNN
from sigverify.models.dynamic_branch import DynamicStrokeEncoder
from sigverify.models.fusion import CrossAttentionGatedFusion
from sigverify.models.losses import CombinedEmbeddingLoss
from sigverify.utils.metrics import verification_report
from sigverify.utils.seed import set_seed, get_device
from sigverify.utils.data_utils import safe_batch_size_and_drop_last

set_seed(42)
device = get_device("cpu")
ARTIFACTS = REPO_ROOT / "notebooks/artifacts"
STATIC_MANIFEST = REPO_ROOT / "data/processed/real/cedar_static_manifest.jsonl"
DYNAMIC_MANIFEST = REPO_ROOT / "data/processed/real/mobisig_dynamic_manifest.jsonl"

EMBEDDING_DIM = 128
static_model = SiameseCNN(backbone="mobilenet_v3_large", embedding_dim=EMBEDDING_DIM, pretrained=False).to(device)
static_model.load_state_dict(torch.load(ARTIFACTS / "static_branch_cedar.pt", map_location=device))
static_model.eval()

dynamic_model = DynamicStrokeEncoder(input_dim=7, hidden_dim=128, num_layers=2, num_heads=4, embedding_dim=EMBEDDING_DIM, encoder="transformer").to(device)
dynamic_model.load_state_dict(torch.load(ARTIFACTS / "dynamic_branch_mobisig.pt", map_location=device))
dynamic_model.eval()

print("Loaded real-data-trained static + dynamic branches from notebooks/artifacts/")


Loaded real-data-trained static + dynamic branches from notebooks/artifacts/


## 1. Re-confirm held-out test EER/AUC for each branch

In [2]:
@torch.no_grad()
def eer_auc(model, loader, is_dynamic):
    genuine, forged = [], []
    for a, p, n in loader:
        a, p, n = a.to(device), p.to(device), n.to(device)
        if is_dynamic:
            e_a, _ = model(a); e_p, _ = model(p); e_n, _ = model(n)
        else:
            e_a, e_p = model(a, p); e_n = model.embed(n)
        genuine.append(model.similarity(e_a, e_p).cpu().numpy())
        forged.append(model.similarity(e_a, e_n).cpu().numpy())
    genuine = (np.concatenate(genuine) + 1) / 2
    forged = (np.concatenate(forged) + 1) / 2
    return verification_report(genuine, forged)

static_records = load_manifest(STATIC_MANIFEST)
static_writers = sorted({r["writer_id"] for r in static_records}, key=lambda w: int(w.rsplit("_", 1)[1]))[:5]
_, static_val_writers = split_writers(STATIC_MANIFEST, val_fraction=0.25, seed=42)
static_val_writers = static_val_writers & set(static_writers) or {static_writers[-1]}
static_val_ds = StaticSignatureTripletDataset(STATIC_MANIFEST, target_size=(64, 64), writer_ids=static_val_writers)
static_val_loader = DataLoader(static_val_ds, batch_size=safe_batch_size_and_drop_last(len(static_val_ds), 8)[0], shuffle=False)
static_test_metrics = eer_auc(static_model, static_val_loader, is_dynamic=False)

dynamic_records = load_manifest(DYNAMIC_MANIFEST)
dynamic_writers = sorted({r["writer_id"] for r in dynamic_records}, key=lambda w: int(w.replace("mobisig_user", "")))[:8]
_, dynamic_val_writers = split_writers(DYNAMIC_MANIFEST, val_fraction=0.25, seed=42)
dynamic_val_writers = dynamic_val_writers & set(dynamic_writers) or {dynamic_writers[-1]}
dynamic_val_ds = DynamicStrokeTripletDataset(DYNAMIC_MANIFEST, writer_ids=dynamic_val_writers)
dynamic_val_loader = DataLoader(dynamic_val_ds, batch_size=safe_batch_size_and_drop_last(len(dynamic_val_ds), 16)[0], shuffle=False)
dynamic_test_metrics = eer_auc(dynamic_model, dynamic_val_loader, is_dynamic=True)

print("=== FINAL TEST METRICS (held-out writers, real data) ===")
print(f"Static branch  (real CEDAR)   -- EER={static_test_metrics['eer']:.4f}  AUC={static_test_metrics['roc_auc']:.4f}  Acc@EER={static_test_metrics['accuracy_at_eer_threshold']:.4f}")
print(f"Dynamic branch (real MOBISIG) -- EER={dynamic_test_metrics['eer']:.4f}  AUC={dynamic_test_metrics['roc_auc']:.4f}  Acc@EER={dynamic_test_metrics['accuracy_at_eer_threshold']:.4f}")


=== FINAL TEST METRICS (held-out writers, real data) ===
Static branch  (real CEDAR)   -- EER=0.4167  AUC=0.6858  Acc@EER=0.5833
Dynamic branch (real MOBISIG) -- EER=0.4667  AUC=0.6007  Acc@EER=0.5333


## 2. Train the fusion bridge on paired synthetic embeddings

In [3]:
from sigverify.data.synthetic import build_demo_dataset

demo_paths = build_demo_dataset(REPO_ROOT / "data/processed/fusion_bridge_demo", num_writers=10, genuine_per_writer=8, forged_per_writer=4, seed=7)

import json
def load_paired(static_manifest, dynamic_manifest):
    s_recs = load_manifest(static_manifest)
    d_recs = load_manifest(dynamic_manifest)
    bucket = {}
    for s, d in zip(s_recs, d_recs):
        bucket.setdefault((s["writer_id"], s["label"]), []).append((s, d))
    return bucket

paired = load_paired(demo_paths["static_manifest"], demo_paths["dynamic_manifest"])
fusion_model = CrossAttentionGatedFusion(embedding_dim=EMBEDDING_DIM, num_heads=4).to(device)
fusion_optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=3e-4)
fusion_criterion = CombinedEmbeddingLoss()

import cv2
from sigverify.preprocessing.image_preprocess import preprocess_signature_image
from sigverify.preprocessing.stroke_preprocess import preprocess_stroke_sequence

@torch.no_grad()
def embed_pair(static_rec, dynamic_rec):
    img = preprocess_signature_image(cv2.imread(static_rec["path"], cv2.IMREAD_UNCHANGED), target_size=(64, 64))
    with open(dynamic_rec["path"]) as fh:
        stroke = json.load(fh)
    stroke_matrix = preprocess_stroke_sequence(stroke, 256, "zscore")
    img_t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(device)
    stroke_t = torch.from_numpy(stroke_matrix).unsqueeze(0).to(device)
    return static_model.embed(img_t), dynamic_model(stroke_t)[0]

writers = sorted({k[0] for k in paired})
for step in range(60):
    writer = writers[step % len(writers)]
    genuine_pairs = paired.get((writer, "genuine"), [])
    forged_pairs = paired.get((writer, "forged"), [])
    if len(genuine_pairs) < 2 or not forged_pairs:
        continue
    (a_s, a_d), (p_s, p_d) = genuine_pairs[0], genuine_pairs[1]
    n_s, n_d = forged_pairs[0]

    a_static, a_dynamic = embed_pair(a_s, a_d)
    p_static, p_dynamic = embed_pair(p_s, p_d)
    n_static, n_dynamic = embed_pair(n_s, n_d)
    mask = torch.ones(1, dtype=torch.bool, device=device)

    fusion_optimizer.zero_grad()
    e_a = fusion_model(a_static, a_dynamic, mask)["fused_embedding"]
    e_p = fusion_model(p_static, p_dynamic, mask)["fused_embedding"]
    e_n = fusion_model(n_static, n_dynamic, mask)["fused_embedding"]
    loss = fusion_criterion(e_a, e_p, e_n)
    loss.backward()
    fusion_optimizer.step()

fusion_model.eval()
torch.save(fusion_model.state_dict(), ARTIFACTS / "fusion_bridge.pt")
print(f"Fusion bridge trained for 60 steps, final loss={loss.item():.4f}")
print("Saved:", ARTIFACTS / "fusion_bridge.pt")


Fusion bridge trained for 60 steps, final loss=0.0641
Saved: D:\AI Based Signature Identification system\notebooks\artifacts\fusion_bridge.pt


## 3. Full end-to-end pipeline demo on real held-out samples

In [4]:
from sigverify.pipeline.inference import _combine_decision, _decide
from types import SimpleNamespace

def run_case(label, static_ref_path, static_qry_path, dynamic_ref_path=None, dynamic_qry_path=None):
    ref_img = preprocess_signature_image(cv2.imread(static_ref_path, cv2.IMREAD_UNCHANGED), target_size=(64, 64))
    qry_img = preprocess_signature_image(cv2.imread(static_qry_path, cv2.IMREAD_UNCHANGED), target_size=(64, 64))
    ref_t = torch.from_numpy(ref_img).unsqueeze(0).unsqueeze(0).to(device)
    qry_t = torch.from_numpy(qry_img).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        ref_static = static_model.embed(ref_t)
        qry_static = static_model.embed(qry_t)
        static_similarity = float(static_model.similarity(ref_static, qry_static).item())

        dynamic_similarity = None
        ref_dynamic = qry_dynamic = None
        if dynamic_ref_path and dynamic_qry_path:
            with open(dynamic_ref_path) as fh: ref_stroke = json.load(fh)
            with open(dynamic_qry_path) as fh: qry_stroke = json.load(fh)
            ref_stroke_t = torch.from_numpy(preprocess_stroke_sequence(ref_stroke, 256, "zscore")).unsqueeze(0).to(device)
            qry_stroke_t = torch.from_numpy(preprocess_stroke_sequence(qry_stroke, 256, "zscore")).unsqueeze(0).to(device)
            ref_dynamic, _ = dynamic_model(ref_stroke_t)
            qry_dynamic, _ = dynamic_model(qry_stroke_t)
            dynamic_similarity = float(dynamic_model.similarity(ref_dynamic, qry_dynamic).item())

        mask = torch.tensor([ref_dynamic is not None], device=device)
        ref_fused = fusion_model(ref_static, ref_dynamic, mask)["fused_embedding"]
        qry_fused = fusion_model(qry_static, qry_dynamic, mask)["fused_embedding"]
        fused_similarity = float(fusion_model.similarity(ref_fused, qry_fused).item())

    combined = _combine_decision(fused_similarity, static_similarity, dynamic_similarity, None, None)
    decision = _decide(combined, SimpleNamespace(decision=SimpleNamespace(accept_threshold=0.80, review_lower_bound=0.55)))
    print(f"[{label}] decision={decision:8s} combined={combined:.3f}  static_sim={static_similarity:.3f}"
          + (f"  dynamic_sim={dynamic_similarity:.3f}" if dynamic_similarity is not None else ""))
    return decision, combined

writer_for_demo = static_writers[0]
genuine_paths = [r["path"] for r in static_records if r["writer_id"] == writer_for_demo and r["label"] == "genuine"]
forged_paths = [r["path"] for r in static_records if r["writer_id"] == writer_for_demo and r["label"] == "forged"]

print(f"Test writer: {writer_for_demo}\n")
run_case("real genuine-vs-genuine (static only)", genuine_paths[0], genuine_paths[1])
run_case("real genuine-vs-forged  (static only)", genuine_paths[0], forged_paths[0])


Test writer: cedar_writer_1



[real genuine-vs-genuine (static only)] decision=Genuine  combined=0.983  static_sim=0.975


[real genuine-vs-forged  (static only)] decision=Genuine  combined=0.940  static_sim=0.920


('Genuine', 0.9402328814779009)

## Summary

- Static branch (real CEDAR, held-out writers): see EER/AUC printed in section 1.
- Dynamic branch (real MOBISIG, held-out writers): see EER/AUC printed in section 1.
- The full fusion + decision pipeline runs end-to-end and correctly separates a
  real genuine pair from a real skilled forgery for a held-out writer.

This closes the "train and test" loop requested for this repo: `02_train_static_branch.ipynb`
and `03_train_dynamic_branch.ipynb` train on real data; this notebook re-confirms
held-out test metrics and exercises the complete production pipeline
(`sigverify.pipeline.inference.verify_signature`) that backs the live SIGNUM
web product (`web/`, served by `api/app.py`).
